<a href="https://colab.research.google.com/github/lcaredda/Research-Project-M2---Plane-Identification-in-PointCloud/blob/main/Script2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROBUSNESS TEST

---



#1) Set up

Libraries

In [48]:
%pip install -q open3d

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import pandas as pd
import random


Visualization function ```plot()```



In [49]:
# @title
def plot(geometries, point_size=2, mesh_opacity=0.5, color_by_height=False, colorscale='Viridis', default_color=(0.7, 0.7, 0.7)):
    """
    Plot Open3D PointClouds and TriangleMeshes in 3D using Plotly,
    with optional coloring by height.

    Parameters:
    - geometries: Open3D geometry or list of geometries (PointCloud / TriangleMesh)
    - point_size: size of points in point clouds
    - mesh_opacity: opacity of meshes
    - color_by_height: if True, color points by z-coordinate; if False, use original colors or default_color
    - colorscale: Plotly colorscale for points (if color_by_height=True)
    - default_color: default RGB color for points if no color is set (e.g., (0.7, 0.7, 0.7) for gray)
    """
    if not isinstance(geometries, list):
        geometries = [geometries]

    graph_objects = []

    for geometry in geometries:
        # --- PointCloud ---
        if isinstance(geometry, o3d.geometry.PointCloud):
            points = np.asarray(geometry.points)
            if points.size == 0:
                continue

            # Color points: prioritize original colors unless color_by_height is True
            if color_by_height:
                colors = points[:, 2]  # Use z-coordinate for color
            elif geometry.has_colors():
                colors = np.asarray(geometry.colors)  # Use original colors
            else:
                colors = np.tile(default_color, (len(points), 1))  # Use default_color for all points

            scatter = go.Scatter3d(
                x=points[:,0], y=points[:,1], z=points[:,2],
                mode='markers',
                marker=dict(
                    size=point_size,
                    color=colors,
                    colorscale=colorscale if color_by_height else None,
                    opacity=0.8,
                    cmin=points[:, 2].min() if color_by_height else None,
                    cmax=points[:, 2].max() if color_by_height else None
                )
            )
            graph_objects.append(scatter)

        # --- TriangleMesh ---
        elif isinstance(geometry, o3d.geometry.TriangleMesh):
            vertices = np.asarray(geometry.vertices)
            triangles = np.asarray(geometry.triangles)
            if vertices.size == 0 or triangles.size == 0:
                continue

            if geometry.has_triangle_normals():
                colors = (0.5, 0.5, 0.5) + np.asarray(geometry.triangle_normals) * 0.5
                colors = tuple(map(tuple, colors))
            else:
                colors = 'lightblue'

            mesh = go.Mesh3d(
                x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
                i=triangles[:,0], j=triangles[:,1], k=triangles[:,2],
                color=colors if isinstance(colors, (np.ndarray, list, tuple)) else colors,
                opacity=mesh_opacity
            )
            graph_objects.append(mesh)

    # --- Layout ---
    fig = go.Figure(
        data=graph_objects,
        layout=dict(
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode='data',
                bgcolor='rgba(0,0,0,0)'
            ),
            paper_bgcolor='rgba(0,0,0,0)',
            margin=dict(l=0, r=0, t=0, b=0)
        )
    )
    fig.show()

# 2) Initialisation

In [50]:
# -----------------------------
# Paramètres du plan
# -----------------------------
a = np.random.uniform(-1, 1)
b = np.random.uniform(-1, 1)
c = np.random.uniform(-1, 1)

error_rate = 0.2   # noise level (σ)
r = 0.1            # LiDAR resolution
A = 20             # plane area (in m²)

L = np.sqrt(A)     # grid length
N = int(((L / r) + 1) ** 2)

print("Number of points:", N)

# -----------------------------
# Génération des points (x, y)
# -----------------------------
x = np.random.uniform(0, L, N)
y = np.random.uniform(0, L, N)

# -----------------------------
# Plan parfait (GROUND TRUTH)
# -----------------------------
z_ref = a * x + b * y + c
points_ref = np.vstack((x, y, z_ref)).T

pcd_ref = o3d.geometry.PointCloud()
pcd_ref.points = o3d.utility.Vector3dVector(points_ref)
pcd_ref.paint_uniform_color([1.0, 1.0, 0.0])  # YELLOW (ground truth)

# Calculate slope angle (in degrees)
slope_rad = np.arctan(np.sqrt(a**2 + b**2))
slope_deg = np.degrees(slope_rad)

print(f"Slope angle (degrees): {slope_deg:.2f}°")

# -----------------------------
# Ajout du bruit (disparité)
# -----------------------------
noise = np.random.normal(0, error_rate, N)
z_noisy = z_ref + noise

points_noisy = np.vstack((x, y, z_noisy)).T

pcd_noise = o3d.geometry.PointCloud()
pcd_noise.points = o3d.utility.Vector3dVector(points_noisy)
pcd_noise.paint_uniform_color([1.0, 0.0, 1.0])  # PINK (noisy)

Number of points: 2090
Slope angle (degrees): 37.61°


PointCloud with 2090 points.

In [51]:
# -----------------------------
# Visualisation
# -----------------------------
plot([pcd_ref, pcd_noise]) #Reference (ground truth) plane in yellow / noise plane in pink

# 3) Detection of a plane inside the noise

**RSPD Algo**

In [52]:
#%% RSPD Planes Detection

# Rename variable for clarity
pcd = pcd_noise

# Ensure normals exist (raise error if not)
# Estimate normals for the point cloud
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))
pcd.orient_normals_towards_camera_location(camera_location=np.array([0, 0, 0])) # Assuming origin as camera location for simplicity

# Detect planar patches using RSPD (default parameters)
oboxes = pcd.detect_planar_patches(
    normal_variance_threshold_deg=30,
    coplanarity_deg=75,
    outlier_ratio=0.75,
    min_plane_edge_length=0.05,
    min_num_points=50,
    search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30)
)

# Prepare geometries for visualization
geometries = []

# Extract points for each plane and create visualization geometries
for obox in oboxes:
    # Get points within the bounding box
    idx = obox.get_point_indices_within_bounding_box(o3d.utility.Vector3dVector(np.asarray(pcd.points)))
    if len(idx) == 0:  # Skip empty planes
        continue

    # Create a thin mesh to represent the plane
    mesh = o3d.geometry.TriangleMesh.create_from_oriented_bounding_box(obox, scale=[1, 1, 0.0001])
    mesh.paint_uniform_color(obox.color)
    geometries.extend([mesh, obox])  # Add both mesh and bounding box

# Add the original point cloud to the visualization
geometries.append(pcd)

# Optional: Print summary of detected planes
print(f"Detected {len(oboxes)} planar patches.")

Detected 0 planar patches.


In [53]:
plot(geometries)

At this step you should remark that the RSPD algo doesn't work on plane detection with noise. The problem probably lie in the RSPD parameter, or maybe the algorithm doesn't work for single plane point cloud.

**RANSAC**

Source for slope computation (link last checked 27/03/2026): https://www.youtube.com/watch?v=snIdXOjUG44

In [54]:
# RANSAC to detect planes (stop if <100 inliers)
remaining_pcd = pcd_noise
rows = []
list_of_obbs = []
i = 0

while True:
    # Detect plane using RANSAC
    plane_model, plane_inliers = remaining_pcd.segment_plane(
        distance_threshold=0.10,
        ransac_n=3,
        num_iterations=1000
    )

    # Stop if plane has ≤100 inliers
    if len(plane_inliers) <= 100:
        break

    # Unpack plane equation: ax + by + cz + d = 0
    a, b, c, d = plane_model

    # Calculate slope angle (arctan(√(a² + b²)/|c|))
    if c == 0:
        slope_angle = 90.0  # Vertical plane
    else:
        slope_rad = np.arctan(np.sqrt(a**2 + b**2) / np.abs(c))
        slope_angle = np.degrees(slope_rad)

    # Log plane data
    rows.append({"Plane n°": i, "Slope (°)": round(slope_angle, 2)})

    # Create OBB for visualization
    plane_points = remaining_pcd.select_by_index(plane_inliers)
    obb_plane = o3d.geometry.OrientedBoundingBox.create_from_points(plane_points.points)
    obb_plane.color = [1, 0, 0]
    list_of_obbs.append(obb_plane)

    # Remove inliers to detect next plane
    remaining_pcd = remaining_pcd.select_by_index(plane_inliers, invert=True)
    i += 1

# Output results
df = pd.DataFrame(rows)
print(df)

# Visualization (optional)
geometries = []
for obb in list_of_obbs:
    plane_mesh = o3d.geometry.TriangleMesh.create_from_oriented_bounding_box(obb, scale=[1, 1, 0.0001])
    plane_mesh.paint_uniform_color(obb.color)
    geometries.extend([plane_mesh, obb])
geometries.append(pcd)

   Plane n°  Slope (°)
0         0      37.53
1         1      37.88
2         2      37.22


In [55]:
plot(geometries)

In [56]:
geometries.append(pcd_ref)
plot(geometries)

The algorithm find multiple planes with similar slope. The problem is the loop, in wich RANSAC algorithm works. It makes it find spurious planes and limit only the number of plane found thanks to a parameter set for the minimum number of points found in a plane.